**Setting Up Dependencies**

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

**Loading and Pre-Analysis**

In [ ]:
ticker = "SPY"
df = yf.download(ticker, start="2022-01-01", end="2026-01-01")

/tmp/ipykernel_1019/934751866.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2022-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed


In [ ]:
df

Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600
...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300


**Feature Engineering (5 inputs)**

In [ ]:
df["PCT_return"] = df["Close"].pct_change()
df

Price,Close,High,Low,Open,Volume,PCT_return
Ticker,SPY,SPY,SPY,SPY,SPY,
Date,,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200,NaN
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700,-0.000335
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900,-0.019202
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900,-0.000939
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600,-0.003954
...,...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600,0.003518
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300,-0.000101


In [ ]:
df["3day_volatility"] = df["PCT_return"].rolling(window=3).std()
df

Price,Close,High,Low,Open,Volume,PCT_return,3day_volatility
Ticker,SPY,SPY,SPY,SPY,SPY,,
Date,,,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200,NaN,NaN
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700,-0.000335,NaN
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900,-0.019202,NaN
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900,-0.000939,0.010723
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600,-0.003954,0.009791
...,...,...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600,0.003518,0.001367
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300,-0.000101,0.002451


In [ ]:
df["volume_norm"] = df["Volume"].rolling(3).mean() / df['Volume'].rolling(20).mean()
df

Price,Close,High,Low,Open,Volume,PCT_return,3day_volatility,volume_norm
Ticker,SPY,SPY,SPY,SPY,SPY,,,
Date,,,,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200,NaN,NaN,NaN
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700,-0.000335,NaN,NaN
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900,-0.019202,NaN,NaN
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900,-0.000939,0.010723,NaN
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600,-0.003954,0.009791,NaN
...,...,...,...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600,0.003518,0.001367,0.746841
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300,-0.000101,0.002451,0.639262


In [ ]:
df["fast_sma"] = df["Close"].rolling(5).mean()
df["slow_sma"] = df["Close"].rolling(20).mean()

In [ ]:
df["Slow_Slope"] = (df["slow_sma"] - df["slow_sma"].shift(3) / (3 * df["slow_sma"].shift(3)))
df["Fast_Slope"] = (df["fast_sma"] - df["fast_sma"].shift(3) / (3 * df["fast_sma"].shift(3)))
df

Price,Close,High,Low,Open,Volume,PCT_return,3day_volatility,volume_norm,fast_sma,slow_sma,Slow_Slope,Fast_Slope
Ticker,SPY,SPY,SPY,SPY,SPY,,,,,,,
Date,,,,,,,,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700,-0.000335,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900,-0.019202,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900,-0.000939,0.010723,NaN,NaN,NaN,NaN,NaN
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600,-0.003954,0.009791,NaN,443.675317,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600,0.003518,0.001367,0.746841,680.033496,677.553928,677.220594,679.700163
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300,-0.000101,0.002451,0.639262,683.183374,678.182211,677.848878,682.850041


In [ ]:
df['DOW-T'] = df.index.dayofweek
df

Price,Close,High,Low,Open,Volume,PCT_return,3day_volatility,volume_norm,fast_sma,slow_sma,Slow_Slope,Fast_Slope,DOW-T
Ticker,SPY,SPY,SPY,SPY,SPY,,,,,,,,
Date,,,,,,,,,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700,-0.000335,NaN,NaN,NaN,NaN,NaN,NaN,1
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900,-0.019202,NaN,NaN,NaN,NaN,NaN,NaN,2
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900,-0.000939,0.010723,NaN,NaN,NaN,NaN,NaN,3
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600,-0.003954,0.009791,NaN,443.675317,NaN,NaN,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600,0.003518,0.001367,0.746841,680.033496,677.553928,677.220594,679.700163,2
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300,-0.000101,0.002451,0.639262,683.183374,678.182211,677.848878,682.850041,4


In [ ]:
real_vol = df["PCT_return"].rolling(10).std()
future_vol = real_vol.shift(-1)
q = future_vol.quantile([1/3, 2/3]).values
df["target_cls"] = np.select([future_vol <= q[0], (future_vol > q[0]) & (future_vol <= q[1]), future_vol > q[1]], [0, 1, 2], default=np.nan).astype('float')

In [ ]:
df

Price,Close,High,Low,Open,Volume,PCT_return,3day_volatility,volume_norm,fast_sma,slow_sma,Slow_Slope,Fast_Slope,DOW-T,target_cls
Ticker,SPY,SPY,SPY,SPY,SPY,,,,,,,,,
Date,,,,,,,,,,,,,,
2022-01-03,449.486450,449.618193,445.854516,448.159751,72668200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
2022-01-04,449.335846,451.622301,447.482234,450.907193,71178700,-0.000335,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
2022-01-05,440.707672,449.740501,440.613574,448.968940,104538900,-0.019202,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN
2022-01-06,440.293671,443.003522,437.931955,440.246636,86858900,-0.000939,0.010723,NaN,NaN,NaN,NaN,NaN,3,NaN
2022-01-07,438.552948,441.479221,437.198022,440.303072,85111600,-0.003954,0.009791,NaN,443.675317,NaN,NaN,NaN,4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,686.730530,687.178163,684.164151,684.313382,39445600,0.003518,0.001367,0.746841,680.033496,677.553928,677.220594,679.700163,2,1.0
2025-12-26,686.660889,688.003728,685.626408,686.989161,41613300,-0.000101,0.002451,0.639262,683.183374,678.182211,677.848878,682.850041,4,0.0


In [ ]:
features = ['3day_volatility', 'volume_norm', 'Slow_Slope', 'Fast_Slope', 'DOW-T']
training_df = df[features + ['target_cls']].dropna()
training_df

Price,3day_volatility,volume_norm,Slow_Slope,Fast_Slope,DOW-T,target_cls
Ticker,,,,,,
Date,,,,,,
2022-02-03,0.018385,0.920992,425.442910,422.846922,3,2.0
2022-02-04,0.017908,0.897985,424.537748,424.117161,4,2.0
2022-02-07,0.014550,0.814718,423.651876,423.618486,0,2.0
2022-02-08,0.005857,0.731445,422.966418,423.240257,1,2.0
2022-02-09,0.009040,0.660152,422.391986,423.276011,2,2.0
...,...,...,...,...,...,...
2025-12-23,0.002272,0.995737,676.357740,675.530705,1,1.0
2025-12-24,0.001367,0.746841,677.220594,679.700163,2,1.0


**Model #1 Training (Logistic Regression)**

In [ ]:
X = training_df[features]
y = training_df['target_cls']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

**Model #1 Analysis (Logistic Regression)**

In [ ]:
log_reg = LogisticRegression(multi_class='multinomial', max_iter=1000,solver='lbfgs')
log_reg.fit(X_train,y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

         0.0       0.52      0.67      0.59        66
         1.0       0.32      0.18      0.23        65
         2.0       0.57      0.65      0.60        65

    accuracy                           0.50       196
   macro avg       0.47      0.50      0.47       196
weighted avg       0.47      0.50      0.48       196



/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


**Model #2 Training  (XGBoost)**

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

X = training_df[features]
y = training_df['target_cls']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)
xgModel = XGBClassifier(
    objective = 'multi:softmax',
    num_class = 3,
    eval_metric = 'mlogloss',
    learning_Rate = 0.05,
    max_depth = 4,
    n_estimators = 200
)
xgModel.fit(X_train, y_train)



/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [00:37:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "learning_Rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_Rate=0.05,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, ...)

**Model #2 Analysis (XGBoost)**

In [ ]:
y_pred = xgModel.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(acc)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.7193877551020408
              precision    recall  f1-score   support

         0.0       0.75      0.80      0.77        66
         1.0       0.62      0.58      0.60        65
         2.0       0.78      0.77      0.78        65

    accuracy                           0.72       196
   macro avg       0.72      0.72      0.72       196
weighted avg       0.72      0.72      0.72       196

[[53 11  2]
 [15 38 12]
 [ 3 12 50]]
